# Metadata schema extraction from documents using Foundation Models (FMs) on Amazon Bedrock
---

In this notebook, we will refer to the data provided in the `data` directory and the metadata schema provided in the [`config.yaml`](config.yaml) file to extract metadata from each document. The prompt that is used to extract metadata from the document is provided in the `prompt_templates` directory. This prompt uses the document and the metadata schema provided by the user to extract relevant data from the documents.

This notebook can be used with custom metadata schemas and prompts, giving users full control and flexibility into metadata extraction from their documents.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import re
import os
import json
import yaml
import time
import boto3
import base64
import globals
import zipfile
import logging
import litellm
import datetime
import requests
from utils import *
from globals import *
from io import BytesIO
from pathlib import Path
from datetime import datetime
from typing import Union, Dict, Optional
from botocore.exceptions import ClientError
from litellm import completion, RateLimitError
from litellm.llms.bedrock.common_utils import BedrockError

### Set a logger

In [3]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers
logger.handlers.clear()

# Add a simple handler
handler = logging.StreamHandler()
formatter = logging.Formatter('[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

In [ ]:
def get_aws_region():
    """Retrieve the AWS region from environment, session, or EC2 metadata."""
    try:
        region: Optional[str] = None
        # Use IMDSv2 to fetch region from EC2 metadata
        token = requests.put(
            EC2_API_TOKEN_URL,
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
            timeout=GET_REGION_TIMEOUT
        ).text
        region = requests.get(
            EC2_METADATA_REGION_URL,
            headers={"X-aws-ec2-metadata-token": token},
            timeout=GET_REGION_TIMEOUT
        ).text
    except requests.RequestException:
        region = None
    return region

region_name = get_aws_region()
# Determine the region from boto3 if the region name is None
region_name = boto3.Session().region_name if region_name is None else region_name
logger.info(f"Setting the region to {region_name}")
logger.info(f"This notebook is being run in region: {region_name}")
# Initialize S3 client
s3_client = boto3.client('s3', region_name=region_name)
# Initialize the bedrock runtime client. This is used to 
# query search results from the KBs
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime', region_name=region_name) 

## Load the config file
---

The config file contains information about the directories, data to extract metadata from, models used in the metadata extraction process, their inference parameters and prompt templates, and the custom metadata schema provided by the user.

In [ ]:
# Get the absolute path to the config file
# This config file contains data about the directory paths, the API specs that
# are used to generate the code, and the agent foundation models that are used to generate the code.
config_data = load_config("config.yaml")

In [ ]:
logger.info(f"metadata schema that is provided by the user:")
print(config_data['metadata_schema'])

# Get Inferences from the FMs on Bedrock
---

In this piece of code, we will generate inferences from the models and get the metadata extracted using the document and the metadata schema provided by the user.

In [7]:
def process_document_with_retry(document_content, schema: str, model_info: dict, file_name: str, doc_source: str = None) -> Dict:
    """
    Process a document to extract metadata using the provided schema and question.
    Supports binary document content (e.g. PDF, DOC, DOCX) by passing a document block
    with a correctly encoded data URL.

    Args:
        document_content: The content of the document to analyze. This may be either a string (for text)
                          or bytes (for binary files like PDF/DOC/DOCX).
        schema: The metadata schema to use for extraction.
        model_info: Dictionary containing model configuration details.
        doc_source: (Optional) The source file name (used to determine MIME type for binary content).

    Returns:
        Dictionary containing the response and execution metadata.
    """
    if model_info.get('prompt') is None:
        logger.error(f"No prompt template found for model information: {model_info}.")

    # Read the prompt template
    prompt_path = Path(os.path.join(config_data['dir_info']['prompt_template_dir'], model_info.get('prompt')))
    prompt = prompt_path.read_text(encoding='utf-8')
    formatted_prompt = prompt.replace("{schema}", schema)

    # If the document content is binary, determine the correct MIME type based on the file extension.
    if isinstance(document_content, bytes):
        # Default to PDF if no doc_source is provided.
        mime_type = "application/pdf"
        if doc_source:
            ext = Path(doc_source).suffix.lower()
            if ext == PDF_EXT:
                mime_type = "application/pdf"
            elif ext == DOC_EXT:
                mime_type = "application/msword"
            elif ext == DOCX_EXT:
                mime_type = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
        # Base64 encode the binary data
        encoded_file = base64.b64encode(document_content).decode("utf-8")
        base64_url = f"data:{mime_type};base64,{encoded_file}"
        
        # Build the message content list using the text prompt and the data URL.
        message_content = [
            {"type": "text", "text": formatted_prompt},
            {"type": "image_url", "image_url": base64_url},
        ]
    else:
        # If the document is text, insert it into the prompt.
        formatted_prompt = prompt.replace("{document}", document_content).replace("{schema}", schema)
        message_content = [
            {"type": "text", "text": formatted_prompt},
        ]
    messages = [
        {
            "role": "user",
            "content": message_content,
        }
    ]
    # Get inference parameters from the model configuration.
    temperature = model_info['inference_parameters'].get('temperature', 0.1)
    max_tokens = model_info['inference_parameters'].get('max_tokens', 500)
    
    ret = {
        "exception": None,
        "prompt": formatted_prompt,
        "completion": None,
        "completion_token_count": None,
        "prompt_token_count": None,
        "model_id": model_info.get('model_id'),
        "file_name": file_name,
        "time_taken_in_seconds": None,
    }
    retry_count = 0
    while True:
        try:
            response = completion(
                model=model_info.get('model_id'),
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            # Extract response content from the API's choices.
            for choice in response.choices:
                if choice.message and choice.message.content:
                    ret["completion"] = choice.message.content.strip()
            # Extract usage statistics.
            ret['prompt_token_count'] = response.usage.prompt_tokens
            ret['completion_token_count'] = response.usage.completion_tokens
            ret['time_taken_in_seconds'] = response._response_ms / 1000
            break
        except (RateLimitError, ClientError, BedrockError) as e:
            is_throttling = (
                (isinstance(e, ClientError) and e.response['Error']['Code'] in ['ThrottlingException', 'TooManyRequestsException'])
                or (isinstance(e, BedrockError) and "Too many requests" in str(e))
                or isinstance(e, RateLimitError)
            )
            if not is_throttling:
                logging.error(f"Unhandled error: {str(e)}")
                ret["exception"] = e
                raise
            retry_count += 1
            wait_time = min(INITIAL_RETRY_DELAY * (2 ** (retry_count - 1)), MAX_RETRY_DELAY)
            logging.warning(
                f"Throttling error encountered: {str(e)}. Retrying in {wait_time:.2f} seconds... (Attempt {retry_count})"
            )
            time.sleep(wait_time)
        except Exception as e:
            logging.error(f"Unexpected exception occurred: {e}")
            ret["exception"] = e
            break
    return ret

In [ ]:
logger.info(f" Models that will be used for this test:")
print(config_data['model_information'])

In [ ]:
logger.info(f"Number of documents to be looped through are {len(config_data['dir_info']['data_files'])}, as follows: {config_data['dir_info']['data_files']}")

In [10]:
from urllib.parse import urlparse

def read_content(file_path_or_url):
    """
    Read content from either a local file or HTTP URL
    """
    # get the parsed version of the file path or url and if it is an 'http' or 'https' link, 
    # then extract the metadata from the document
    parsed = urlparse(str(file_path_or_url))
    # Check if it's an HTTP(S) URL
    if parsed.scheme in ('http', 'https'):
        logger.info(f"File is a URL: {file_path_or_url}, reading the content")
        try:
            response = requests.get(file_path_or_url, timeout=30)
            response.raise_for_status()  # Raise an exception for bad status codes
            return response.content
        except requests.exceptions.RequestException as e:
            raise Exception(f"Error fetching URL {file_path_or_url}: {str(e)}")
    else:
        try:
            with open(file_path_or_url, 'rb') as f:
                return f.read()
        except IOError as e:
            raise Exception(f"Error reading file {file_path_or_url}: {str(e)}")


In [ ]:
base_results_dir = Path(config_data['dir_info']['results_dir'])
base_results_dir.mkdir(parents=True, exist_ok=True)

# Get list of documents (reading each file as binary)
data_dir = Path(config_data['dir_info']['source_dir'])
documents = []
for file_name in config_data['dir_info']['data_files']:
    try:
        # Determine if it's a URL or local path
        if file_name.startswith((HTTP_PREFIX, HTTPS_PREFIX)):
            file_path = file_name  # Use URL as is
        else:
            file_path = data_dir / file_name  # Local file path
        content = read_content(file_path)
        documents.append({
            'source': file_name,
            'document_content': content,
            'file_name': file_name
        })
    except Exception as e:
        print(f"Error processing {file_name}: {str(e)}")
        continue

# Process each model
for model_info in config_data['model_information']:
    if not model_info.get('run_test', True):
        continue
    logger.info(f"====================Running inference for model: {model_info.get('model_id')}=====================================")
    model_id = model_info['model_id']
    logging.info(f"\nProcessing with model: {model_id}")
    model_dir = base_results_dir / model_id.replace('.', '-').replace('/', '_')
    model_dir.mkdir(exist_ok=True)
    
    # Process each document
    for doc in documents:
        doc_source = doc['source']
        logging.info(f"Processing document: {doc_source}")
        try:
            result = process_document_with_retry(
                document_content=doc['document_content'],
                schema=config_data['metadata_schema'],
                model_info=model_info, 
                doc_source=doc['source'], 
                file_name=doc['file_name']
            )
            # Create a filename based on the document source
            filename = f"{Path(doc_source).stem}.json"
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{Path(doc_source).stem}_{timestamp}.json"
            result_path = model_dir / filename
            with open(result_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)

            logging.info(f"Successfully processed {doc_source}")
        except Exception as e:
            logging.error(f"Error processing {doc_source} with {model_id}: {str(e)}")
            # Save error result
            error_result = {
                'source': doc_source,
                'model_id': model_id,
                'error': str(e),
            }
            error_filename = f"{Path(doc_source).stem}_error.json"
            error_path = model_dir / error_filename
            with open(error_path, 'w', encoding='utf-8') as f:
                json.dump(error_result, f, indent=2, ensure_ascii=False)
